<hr>

# 🧫 DATA CLEANING - Valeurs Foncieres


<style>
h1 {
    text-align: left;
    color: blue;
    font-weight: bold;
}

</style>
<hr>

<style>
h3 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

### 📂 IMPORTs

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200) # to display all columns in the dataframe
print("✅ Libraries imported successfully")

✅ Libraries imported successfully


<hr>

## 0️⃣ AGGREGATE DATASETS

<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

- Aggregation of 6 TXT FILES INTO 1 CSV
- Fixed text accents & added INSEE, No messy text thanks to:
    - encoding latin-1
    - decoding utf-8

In [3]:
import pandas as pd

# -------------------------
# FILES
# -------------------------
dvf_files = [
    "../data/raw/ValeursFoncieres-2020-S2.txt",
    "../data/raw/ValeursFoncieres-2021.txt",
    "../data/raw/ValeursFoncieres-2022.txt",
    "../data/raw/ValeursFoncieres-2023.txt",
    "../data/raw/ValeursFoncieres-2024.txt",
    "../data/raw/ValeursFoncieres-2025-S1.txt"
]

# -------------------------
# COLUMNS TO KEEP
# -------------------------
usecols = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Commune",
    "Type local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Nombre de lots",
    "Surface terrain",
    "No voie",
    "Code voie",
    "B/T/Q",
    "Voie",
    "Code postal",
    "Code departement",
    "Code commune"
]

# -------------------------
# OUTPUT
# -------------------------
output_file = "../data/processed/00_RAW_ValeursFoncieres_2020-S2_2025-S1.csv"

# -------------------------
# DTYPES ON READ
# -------------------------
dtype_dict = {
    "Date mutation": str,
    "Nature mutation": str,
     # "Valeur fonciere": float,
    "No voie": str,
    "Code voie": str,
    "B/T/Q": str,
    "Voie": str,
    "Code postal": str,
    "Commune": str,
    "Code departement": str,
    "Code commune": str,
     # "Nombre de lots": int,
    "Type local": str,
     # "Surface reelle bati": float,
     # "Nombre pieces principales": int,
     # "Surface terrain": float
}

# -------------------------
# TEXT COLUMNS TO FIX
# -------------------------
text_cols = [
    "Nature mutation",
    "Voie",
    "Commune",
    "Type local"
]

# -------------------------
# ENCODING FIX
# -------------------------
def fix_encoding(series):
    series = series.copy()
    mask = series.notna()
    series.loc[mask] = (
        series.loc[mask]
        .str.encode("latin-1", errors="ignore")
        .str.decode("utf-8", errors="ignore")
        .str.strip()
    )
    return series

# -------------------------
# CLEAN STRING CODES
# removes trailing .0 before converting/keeping as string
# -------------------------
def clean_code_str(series, zfill=None):
    series = series.copy()
    mask = series.notna()

    series.loc[mask] = (
        series.loc[mask]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

    series = series.astype("string")

    if zfill is not None:
        mask2 = series.notna()
        series.loc[mask2] = series.loc[mask2].str.zfill(zfill)

    return series

# -------------------------
# PROCESSING
# -------------------------
with open(output_file, "w", newline="", encoding="utf-8") as f_out:
    header_written = False

    for file in dvf_files:
        print(f"Processing: {file}")

        for chunk in pd.read_csv(
            file,
            sep="|",
            usecols=usecols,
            chunksize=50_000,
            encoding="latin-1",
            decimal=",",
            dtype=dtype_dict,
            low_memory=False
        ):
            # -------------------------
            # FIX TEXT ENCODING
            # -------------------------
            for col in text_cols:
                chunk[col] = fix_encoding(chunk[col])

            # -------------------------
            # CLEAN "No voie" -> string, no .0
            # -------------------------
            chunk["No voie"] = clean_code_str(chunk["No voie"])

            # -------------------------
            # CLEAN POSTAL CODE -> string, 5 digits
            # -------------------------
            chunk["Code postal"] = clean_code_str(chunk["Code postal"], zfill=5)

            # -------------------------
            # CLEAN DEPARTMENT CODE -> string
            # 01 to 95  -> 2 digits (mainland France)
            # 2A / 2B stay unchanged (mainland Corsica)
            # 971 to 976 -> 3 digits (overseas departments)
            
            # -------------------------
            chunk["Code departement"] = clean_code_str(chunk["Code departement"])

            mask_dep = chunk["Code departement"].notna()

            # Corsica codes stay as-is
            mask_corsica = chunk["Code departement"].isin(["2A", "2B"])

            # Overseas departments: 971, 972, 973, 974, 976 -> 3 digits
            mask_overseas = chunk["Code departement"].str.startswith("97", na=False)

            # Metropolitan numeric departments except Corsica -> 2 digits
            mask_numeric = chunk["Code departement"].str.fullmatch(r"\d+", na=False)
            mask_metro = mask_dep & mask_numeric & ~mask_overseas & ~mask_corsica

            chunk.loc[mask_metro, "Code departement"] = (
                chunk.loc[mask_metro, "Code departement"].str.zfill(2)
            )

            chunk.loc[mask_overseas, "Code departement"] = (
                chunk.loc[mask_overseas, "Code departement"].str.zfill(3)
            )

            # -------------------------
            # CLEAN COMMUNE CODE -> string
            # 01 to 95, 2A, 2B -> 3 digits
            # 97#               -> 2 digits
            # -------------------------
            chunk["Code commune"] = clean_code_str(chunk["Code commune"])

            mask_com = chunk["Code commune"].notna()
            mask_97 = chunk["Code departement"].str.startswith("97", na=False)

            chunk.loc[mask_com & mask_97, "Code commune"] = (
                chunk.loc[mask_com & mask_97, "Code commune"].str.zfill(2)
            )
            chunk.loc[mask_com & ~mask_97, "Code commune"] = (
                chunk.loc[mask_com & ~mask_97, "Code commune"].str.zfill(3)
            )

            # ----------------------------------
            # CLEAN VOIE CODE -> string, no .0
            # ----------------------------------
            chunk["Code voie"] = clean_code_str(chunk["Code voie"])

            # -------------------------
            # Data Types
            # -------------------------
            chunk["Voie"] = chunk["Voie"].astype("object")
            chunk["Commune"] = chunk["Commune"].astype("object")
            chunk["Nature mutation"] = chunk["Nature mutation"].astype("object")
            chunk["B/T/Q"] = chunk["B/T/Q"].astype("object")
            chunk["Code postal"] = chunk["Code postal"].astype("object")
            chunk["Code departement"] = chunk["Code departement"].astype("object")
            chunk["Code commune"] = chunk["Code commune"].astype("object")
            chunk["Code voie"] = chunk["Code voie"].astype("object")
            chunk["No voie"] = chunk["No voie"].astype("object")


            # -------------------------
            # CREATE FULL INSEE CODE
            # -------------------------
            # INSEE format:
            # - Mainland (01–95, 2A, 2B): insee_code = DD + CCC  → 5 characters
            # - Overseas (971–976):       insee_code = DDD + CC → 5 characters
            # 
            # Note:
            # - Code departement already cleaned (01–95, 2A/2B, 971–976)
            # - Code commune already formatted:
            #     * 3 digits for mainland
            #     * 2 digits for overseas
            # 
            # So we can safely concatenate

            #mask_insee = chunk["Code departement"].notna() & chunk["Code commune"].notna()

            #chunk["insee_code"] = (
            #    chunk["Code departement"]
            #    .astype("string")
            #    .str.cat(chunk["Code commune"].astype("string"))
            #)

            # ensure missing values remain <NA>
            #chunk.loc[~mask_insee, "insee_code"] = pd.NA

            # -------------------------
            # NUMERIC CONVERSION
            # -------------------------
            chunk["Valeur fonciere"] = pd.to_numeric(chunk["Valeur fonciere"], errors="coerce")
            chunk["Surface reelle bati"] = pd.to_numeric(chunk["Surface reelle bati"], errors="coerce")
            chunk["Surface terrain"] = pd.to_numeric(chunk["Surface terrain"], errors="coerce")

            chunk["Nombre de lots"] = (
                pd.to_numeric(chunk["Nombre de lots"], errors="coerce")
                .round()
                .astype("Int64")
            )
            chunk["Nombre pieces principales"] = (
                pd.to_numeric(chunk["Nombre pieces principales"], errors="coerce")
                .round()
                .astype("Int64")
            )

            # -------------------------
            # MEMORY OPTIMIZATION
            # -------------------------
            chunk["Type local"] = chunk["Type local"].astype("category")

            # -------------------------
            # WRITE CLEAN UTF-8 CSV
            # -------------------------
            chunk.to_csv(
                f_out,
                index=False,
                header=not header_written,
                encoding="utf-8"
            )

            header_written = True
            del chunk

print("✅ Encoding fixed + CSV saved in clean UTF-8")
print(f"✅ CSV File saved at {output_file}")

Processing: ../data/raw/ValeursFoncieres-2020-S2.txt
Processing: ../data/raw/ValeursFoncieres-2021.txt
Processing: ../data/raw/ValeursFoncieres-2022.txt
Processing: ../data/raw/ValeursFoncieres-2023.txt
Processing: ../data/raw/ValeursFoncieres-2024.txt
Processing: ../data/raw/ValeursFoncieres-2025-S1.txt
✅ Encoding fixed + CSV saved in clean UTF-8
✅ CSV File saved at ../data/processed/00_RAW_ValeursFoncieres_2020-S2_2025-S1.csv


### **RENAME COLUMNS**

In [2]:
import pandas as pd

input_file = "../data/processed/00_RAW_ValeursFoncieres_2020-S2_2025-S1.csv"
output_file = "../data/processed/01_STANDARD_ValeursFoncieres_2020-S2_2025-S1.csv"

dtype_dict = {
    "Date mutation": "string",
    "Nature mutation": "string",
    "Code voie": "string",
    "No voie": "string",
    "Voie": "string",
    "B/T/Q": "string",
    "Code postal": "string",
    "Commune": "string",
    "Code departement": "string",
    "Code commune": "string",
    "Type local": "string"
}


rename_columns = {
    "Date mutation": "transaction_date",
    "Nature mutation": "transaction_type",
    "Valeur fonciere": "property_value",
    "No voie": "street_number",
    "B/T/Q": "btq_code",
    "Voie": "street_name",
    "Code postal": "postal_code",
    "Commune": "com_name",
    "Nombre de lots": "lots_count",
    "Type local": "property_type",
    "Surface reelle bati": "building_real_surface",
    "Nombre pieces principales": "main_rooms_count",
    "Surface terrain": "land_surface",
    "Code commune": "com_code",
    "Code departement": "dep_code",
    "Code voie": "street_code"
}


with open(output_file, "w", newline="", encoding="utf-8") as f_out:
    header_written = False

    for chunk in pd.read_csv(
        input_file,
        chunksize=100_000,
        dtype=dtype_dict,
        encoding="utf-8",
        low_memory=False
    ):
        chunk.rename(columns=rename_columns, inplace=True)

        chunk.to_csv(
            f_out,
            index=False,
            header=not header_written,
            encoding="utf-8"
        )

        header_written = True

print(f"✅ Standardized CSV saved to {output_file}")

✅ Standardized CSV saved to ../data/processed/01_STANDARD_ValeursFoncieres_2020-S2_2025-S1.csv


### **Function Load DataFrame - Valeurs Foncieres**

In [1]:
import pandas as pd

# -------------------------
# LOAD STANDARDIZED CSV
# -------------------------
df = pd.read_csv("../data/processed/01_STANDARD_ValeursFoncieres_2020-S2_2025-S1.csv", sep=",", encoding="utf-8", low_memory=False)


ParserError: Error tokenizing data. C error: out of memory

### **EXPLORATION**

In [ ]:
import pandas as pd

# -------------------------
# SHAPE & HEAD
# -------------------------
print("STANDARDIZED Valeurs Foncieres:")
print("Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")
display(df.head())

# -------------------------
# INFO & UNIQUE VALUES
# -------------------------
# Display data types & missing values of each column in df
print("Data Types & Missing Values of Each Column:")
display(df.info())
print(100 * "-")

# Display unique values of each column in df
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"➡️ {col} ({len(unique_vals)}) 🧾 unique values:")
    print(f"{unique_vals}")
    print(100 * "-")



| Original Column Name          | Standardized              | Description                                      | Type (Cat/Numeric) | Python Type        |
|-----------------------------|--------------------------|--------------------------------------------------|--------------------|--------------------|
| **Date mutation**           | `transaction_date`       | Date when the property was sold                  | Categorical        | string             |
| **Nature mutation**         | `transaction_type`       | Nature of transaction (sale, donation, etc.)     | Categorical        | string             |
| **Valeur fonciere**         | `property_value`         | Transaction price in euros                       | Numeric (TARGET)            | float64            |
| **No voie**                 | `street_number`          | Street number of the property                    | Categorical        | string             |
| **B/T/Q**                   | `btq_code`               | Bâtiment/Type/Quartier code                     | Categorical        | string             |
| **Voie**                    | `street_name`            | Full street name                                 | Categorical        | string             |
| **Code postal**             | `postal_code`            | Postal code (categorical, not numeric)           | Categorical        | string             |
| **Commune**                 | `com_name`               | Commune name                                     | Categorical        | string             |
| **Nombre de lots**          | `lots_count`             | Number of lots in the property                   | Numeric            | Int64 (nullable)   |
| **Type local**              | `property_type`          | Property type (Apartment, House, etc.)           | Categorical        | category           |
| **Surface reelle bati**     | `building_real_surface`  | Built area in square meters                      | Numeric            | float64            |
| **Nombre pieces principales**| `main_rooms_count`      | Number of main rooms                             | Numeric            | Int64 (nullable)   |
| **Surface terrain**         | `land_surface`           | Land area in square meters                       | Numeric            | float64            |
| **Code departement**        | `dep_code`               | Department code (DD or DDD)                      | Categorical        | string             |
| **Code commune**            | `com_code`               | Commune code (CCC or CC)                         | Categorical        | string             |
| **Code voie**               | `street_id`              | Street identifier                               | Categorical        | string             |